# **4.1 — Maximização de Lucro da FitStore**

In [ ]:

import pulp

# 4.1 - Maximizacao de Lucro da FitStore

# Criando o modelo matematico
modelo = pulp.LpProblem("Lucro_FitStore", pulp.LpMaximize)

# Variaveis: quantidade de produtos
whey = pulp.LpVariable("Whey", lowBound=0, cat="Integer")
creatina = pulp.LpVariable("Creatina", lowBound=0, cat="Integer")

# Funcao objetivo: maximizar o lucro
modelo += 40 * whey + 30 * creatina, "Lucro_Total"

# Restricao 1: orcamento de R$ 2.000
modelo += 80 * whey + 50 * creatina <= 2000, "Orcamento"

# Restricao 2: capacidade de estoque
modelo += 3 * whey + 2 * creatina <= 60, "Estoque"

# Resolver o problema
modelo.solve(pulp.PULP_CBC_CMD(msg=False))

# Mostrar os resultados
print("Status:", pulp.LpStatus[modelo.status])
print("Quantidade de Whey:", int(whey.varValue))
print("Quantidade de Creatina:", int(creatina.varValue))
print("Lucro maximo: R$", pulp.value(modelo.objective))


Status: Optimal
Quantidade de Whey: 0
Quantidade de Creatina: 30
Lucro maximo: R$ 900.0


## **4.1.1 — Análise do Cenário de Crise**

In [ ]:

import pulp

# Cenario de crise da FitStore
modelo_crise = pulp.LpProblem("Crise_FitStore", pulp.LpMaximize)

whey = pulp.LpVariable("Whey_Crise", lowBound=0, cat="Integer")
creatina = pulp.LpVariable("Creatina_Crise", lowBound=0, cat="Integer")

# Maximizar o lucro
modelo_crise += 40 * whey + 30 * creatina

# Orçamento reduzido para R$ 1.200
modelo_crise += 80 * whey + 50 * creatina <= 1200

# Estoque continua com capacidade de 60
modelo_crise += 3 * whey + 2 * creatina <= 60

# Resolver
modelo_crise.solve(pulp.PULP_CBC_CMD(msg=False))

print("Status:", pulp.LpStatus[modelo_crise.status])
print("Quantidade de Whey:", int(whey.varValue))
print("Quantidade de Creatina:", int(creatina.varValue))
print("Lucro maximo na crise: R$", pulp.value(modelo_crise.objective))


Status: Optimal
Quantidade de Whey: 0
Quantidade de Creatina: 24
Lucro maximo na crise: R$ 720.0


### Análise do Cenário de Crise da FitStore

Com a crise, o orçamento da FitStore precisou ser reduzido de R\$ 2.000 para R\$ 1.200, afetando diretamente o lucro máximo da empresa. A quantidade de creatina também diminuiu de 30 para 24 unidades, fazendo com que o lucro caísse de R\$ 900 para R\$ 720.

Para tentar recuperar parte desse lucro, a FitStore poderia aumentar um pouco o preço da creatina, mas seria necessário analisar se esse aumento não afastaria os clientes e prejudicaria as vendas.

Essa simulação é importante porque ajuda a empresa a analisar a quantidade de produtos que pode comprar com o orçamento disponível e qual opção pode trazer o maior lucro. Dessa forma, a FitStore consegue se planejar melhor e evitar decisões que possam prejudicar seu dinheiro e estoque.

## **4.2 — Minimização de Custos da FitStore**

In [ ]:

import pulp

# Centros de distribuição e lojas
centros = ["SP", "RJ"]
lojas = ["Loja1", "Loja2", "Loja3"]

# Quantidade disponível em cada centro
oferta = {
    "SP": 50,
    "RJ": 40
}

# Quantidade necessária em cada loja
demanda = {
    "Loja1": 30,
    "Loja2": 25,
    "Loja3": 35
}

# Custo de transporte por unidade
custos = {
    "SP": {"Loja1": 5, "Loja2": 8, "Loja3": 12},
    "RJ": {"Loja1": 10, "Loja2": 6, "Loja3": 7}
}

# Criar modelo de minimização
modelo = pulp.LpProblem("Minimizacao_Custos_FitStore", pulp.LpMinimize)

# Quantidade enviada de cada centro para cada loja
rotas = pulp.LpVariable.dicts(
    "Envio", (centros, lojas), lowBound=0, cat="Integer"
)

# Função objetivo: minimizar custo total
modelo += pulp.lpSum(
    custos[i][j] * rotas[i][j]
    for i in centros for j in lojas
)

# Restrição: não ultrapassar a oferta de cada centro
for i in centros:
    modelo += pulp.lpSum(rotas[i][j] for j in lojas) <= oferta[i]

# Restrição: atender exatamente a demanda de cada loja
for j in lojas:
    modelo += pulp.lpSum(rotas[i][j] for i in centros) == demanda[j]

# Resolver o problema
modelo.solve(pulp.PULP_CBC_CMD(msg=False))

# Mostrar os resultados
print("Status:", pulp.LpStatus[modelo.status])

if pulp.LpStatus[modelo.status] == "Optimal":
    for i in centros:
        for j in lojas:
            print(f"{i} -> {j}: {int(pulp.value(rotas[i][j]))} unidades")

    print("Custo mínimo total: R$", pulp.value(modelo.objective))


Status: Optimal
SP -> Loja1: 30 unidades
SP -> Loja2: 20 unidades
SP -> Loja3: 0 unidades
RJ -> Loja1: 0 unidades
RJ -> Loja2: 5 unidades
RJ -> Loja3: 35 unidades
Custo mínimo total: R$ 585.0


In [ ]:

# 4.2 - Cenario What-If: aumento do frete

# Copiar os custos originais
custos_crise = {
    i: custos[i].copy() for i in centros
}

# Aumento do frete do RJ para a Loja 3
custos_crise["RJ"]["Loja3"] = 11

# Criar novo modelo
modelo_crise = pulp.LpProblem(
    "Custos_Crise_FitStore", pulp.LpMinimize
)

# Variaveis de decisao
rotas_crise = pulp.LpVariable.dicts(
    "Envio_Crise", (centros, lojas),
    lowBound=0, cat="Integer"
)

# Minimizar o custo de transporte
modelo_crise += pulp.lpSum(
    custos_crise[i][j] * rotas_crise[i][j]
    for i in centros for j in lojas
)

# Limite de produtos em cada centro
for i in centros:
    modelo_crise += pulp.lpSum(
        rotas_crise[i][j] for j in lojas
    ) <= oferta[i]

# Atender a demanda de cada loja
for j in lojas:
    modelo_crise += pulp.lpSum(
        rotas_crise[i][j] for i in centros
    ) == demanda[j]

# Resolver
modelo_crise.solve(pulp.PULP_CBC_CMD(msg=False))

# Mostrar resultados
print("Status:", pulp.LpStatus[modelo_crise.status])

if pulp.LpStatus[modelo_crise.status] == "Optimal":
    for i in centros:
        for j in lojas:
            print(
                f"{i} -> {j}: "
                f"{int(pulp.value(rotas_crise[i][j]))} unidades"
            )

    print(
        "Custo minimo na crise: R$",
        pulp.value(modelo_crise.objective)
    )


Status: Optimal
SP -> Loja1: 30 unidades
SP -> Loja2: 0 unidades
SP -> Loja3: 20 unidades
RJ -> Loja1: 0 unidades
RJ -> Loja2: 25 unidades
RJ -> Loja3: 15 unidades
Custo minimo na crise: R$ 705.0


### **Análise do Cenário de Crise da FitStore**

No cenário de crise, o custo mínimo de transporte da FitStore aumentou de R\$ 585,00 para R\$ 705,00, porque os valores das entregas dos centros de distribuição até as lojas ficaram mais caros.

Para tentar diminuir esses gastos, a FitStore poderia recalcular as rotas de entrega, procurando as opções com menor custo. Mesmo assim, a empresa ainda gastaria mais do que no cenário normal.

Por isso, é importante fazer essa simulação antes de tomar decisões, pois ela ajuda a empresa a ter uma noção dos gastos e comparar as melhores opções de entrega. Concluí que, mesmo em um cenário de crise, a FitStore precisa buscar as rotas mais econômicas para evitar gastos desnecessários.

### **4.3 — Análise Crítica (Otimização)**

Os resultados que conseguimos no Python fazem sentido para a FitStore, porque mostram quanto a empresa pode lucrar e gastar de acordo com os recursos disponíveis. Na maximização de lucro, quando o orçamento foi reduzido de R\$ 2.000,00 para R\$ 1.200,00, o lucro máximo também diminuiu de R\$ 900,00 para R\$ 720,00. Isso aconteceu porque a empresa passou a ter menos dinheiro disponível para investir na compra dos produtos. Já na minimização de custos, o aumento do frete fez os gastos com transporte passarem de R\$ 585,00 para R\$ 705,00, ou seja, a FitStore precisaria gastar R\$ 120,00 a mais para realizar as entregas.

Essas simulações são importantes porque ajudam a FitStore a analisar os riscos e os gastos que pode enfrentar antes de tomar decisões. Com a análise de cenários (What-If), a empresa consegue entender o que pode acontecer se o orçamento diminuir ou se o transporte ficar mais caro. Dessa forma, a gerência pode se preparar melhor para imprevistos, planejar a compra dos produtos e escolher as rotas de entrega com menor custo. Isso ajuda a evitar gastos desnecessários, reduzir possíveis prejuízos e tomar decisões mais seguras no dia a dia.